# 08 — Modelagem e Validação de Machine Learning

## 1. Objetivo da etapa

Este notebook tem como objetivo desenvolver, treinar, validar e comparar modelos supervisionados de Machine Learning para prever se um aluno será considerado alfabetizado ou não alfabetizado.

A modelagem será realizada a partir dos conjuntos de treino, validação e teste definidos e persistidos na etapa anterior, preservando a separação territorial por município e as decisões adotadas para prevenção de data leakage.

O pré-processamento será integrado diretamente às pipelines dos modelos por meio da arquitetura reutilizável definida em `src/preprocessing.py`, contemplando:

- engenharia de atributos por meio de indicadores de ausência;
- imputação de valores ausentes nas variáveis numéricas;
- transformação das variáveis categóricas;
- padronização das variáveis numéricas quando requerida pela família do modelo;
- integração entre pré-processamento e estimador.

O conjunto de treino será utilizado para o aprendizado dos parâmetros dos modelos e das transformações associadas.

O conjunto de validação será utilizado para comparação de alternativas, análise das métricas, seleção de modelos e demais decisões da etapa de desenvolvimento.

O conjunto de teste permanecerá isolado durante esse processo e será utilizado somente após a definição da estratégia final, permitindo uma avaliação independente da capacidade de generalização do modelo selecionado.

## 2. Carregamento dos conjuntos de modelagem

Os conjuntos de treino, validação e teste foram definidos e persistidos na etapa anterior do projeto.

Nesta etapa, esses artefatos serão carregados diretamente do diretório `tech_challenge_fase3/modelagem`, preservando as partições territoriais já estabelecidas e evitando a repetição do processo de separação dos dados.

Cada arquivo preditivo contém as 16 features selecionadas para modelagem e o target `in_alfabetizado`.

Também serão carregados os arquivos auxiliares de rastreabilidade gerados no Notebook 07. Esses arquivos preservam `id_aluno`, `id_escola` e `co_municipio` e permanecem completamente separados das features utilizadas pelos modelos.

Após o carregamento, serão verificadas as dimensões e a correspondência estrutural entre os conjuntos preditivos e seus respectivos arquivos auxiliares.


In [0]:
# Objetivo:
#
# Carregar os conjuntos de treino, validação
# e teste persistidos na etapa anterior.
#
# Justificativa:
#
# A utilização dos artefatos já definidos e
# validados garante que a modelagem utilize
# exatamente as mesmas partições territoriais
# estabelecidas no Notebook 07.
#
# Dessa forma, não é necessário repetir o
# processo de separação dos dados.
#
# Ação:
#
# Carrega os três arquivos Parquet produzidos
# na Fase 3 e verifica suas dimensões.

import pandas as pd

FASE3_ROOT = (
    "/Volumes/workspace/default/vol_trio_drive/"
    "projetos/fiap/tech_challenge_fase3"
)

MODELAGEM_PATH = f"{FASE3_ROOT}/modelagem"

df_treino = pd.read_parquet(
    f"{MODELAGEM_PATH}/treino.parquet"
)

df_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/validacao.parquet"
)

df_teste = pd.read_parquet(
    f"{MODELAGEM_PATH}/teste.parquet"
)


pd.Series({
    "registros_treino": df_treino.shape[0],
    "colunas_treino": df_treino.shape[1],
    "registros_validacao": df_validacao.shape[0],
    "colunas_validacao": df_validacao.shape[1],
    "registros_teste": df_teste.shape[0],
    "colunas_teste": df_teste.shape[1]
})

### 2.1 Carregamento das variáveis auxiliares de rastreabilidade

Além dos conjuntos preditivos, serão carregados os arquivos auxiliares gerados no Notebook 07.

Esses arquivos preservam `id_aluno`, `id_escola` e `co_municipio`, permitindo associar posteriormente as previsões dos modelos às observações originais.

As variáveis auxiliares não serão utilizadas como features e permanecerão fora das pipelines de Machine Learning.

Sua finalidade é exclusivamente garantir rastreabilidade e apoiar análises posteriores por aluno, escola e município.


In [0]:
# Objetivo:
#
# Carregar as variáveis auxiliares de
# rastreabilidade geradas no Notebook 07.
#
# Justificativa:
#
# Os identificadores de aluno, escola e município
# não participam do treinamento dos modelos, mas
# precisam permanecer associados às observações
# para permitir a rastreabilidade das previsões.
#
# Ação:
#
# Carrega os três arquivos auxiliares correspondentes
# aos conjuntos de treino, validação e teste.

auxiliares_treino = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_treino.parquet"
)

auxiliares_validacao = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_validacao.parquet"
)

auxiliares_teste = pd.read_parquet(
    f"{MODELAGEM_PATH}/auxiliares_teste.parquet"
)


pd.Series({
    "auxiliares_treino": auxiliares_treino.shape,
    "auxiliares_validacao": auxiliares_validacao.shape,
    "auxiliares_teste": auxiliares_teste.shape
})


### 2.2 Validação da correspondência entre dados preditivos e auxiliares

Os arquivos auxiliares foram persistidos no Notebook 07 utilizando exatamente a mesma ordem de linhas dos respectivos conjuntos preditivos.

Nesta etapa será validado o contrato estrutural dessa persistência: quantidade de registros, índices após a leitura, presença das três colunas auxiliares esperadas e ausência desses identificadores nos conjuntos utilizados pelos modelos.

A correspondência semântica entre as linhas foi estabelecida no Notebook 07 antes da persistência; aqui é verificado que essa estrutura permaneceu íntegra após a leitura dos Parquets.


In [0]:
# Objetivo:
#
# Validar a correspondência estrutural entre os
# conjuntos preditivos e os arquivos auxiliares.
#
# Justificativa:
#
# As previsões só podem ser associadas aos alunos
# se os arquivos auxiliares preservarem exatamente
# a mesma quantidade e ordem de registros definida
# no Notebook 07.
#
# Também é necessário confirmar que os identificadores
# permanecem fora dos dados utilizados pelos modelos.
#
# Ação:
#
# Verifica dimensões, índices, colunas esperadas e
# ausência das variáveis auxiliares nos DataFrames
# preditivos.

colunas_auxiliares_esperadas = {
    "id_aluno",
    "id_escola",
    "co_municipio"
}

validacao_rastreabilidade = pd.Series({
    "treino_mesmo_numero_registros": (
        len(df_treino) == len(auxiliares_treino)
    ),
    "validacao_mesmo_numero_registros": (
        len(df_validacao) == len(auxiliares_validacao)
    ),
    "teste_mesmo_numero_registros": (
        len(df_teste) == len(auxiliares_teste)
    ),
    "treino_indices_alinhados": (
        df_treino.index.equals(auxiliares_treino.index)
    ),
    "validacao_indices_alinhados": (
        df_validacao.index.equals(auxiliares_validacao.index)
    ),
    "teste_indices_alinhados": (
        df_teste.index.equals(auxiliares_teste.index)
    ),
    "colunas_auxiliares_treino_ok": (
        set(auxiliares_treino.columns)
        == colunas_auxiliares_esperadas
    ),
    "colunas_auxiliares_validacao_ok": (
        set(auxiliares_validacao.columns)
        == colunas_auxiliares_esperadas
    ),
    "colunas_auxiliares_teste_ok": (
        set(auxiliares_teste.columns)
        == colunas_auxiliares_esperadas
    ),
    "auxiliares_fora_dos_dados_preditivos": (
        colunas_auxiliares_esperadas.isdisjoint(df_treino.columns)
        and colunas_auxiliares_esperadas.isdisjoint(df_validacao.columns)
        and colunas_auxiliares_esperadas.isdisjoint(df_teste.columns)
    )
})

validacao_rastreabilidade


### 2.3 Separação entre features e target

Após o carregamento dos conjuntos persistidos, será realizada a separação entre as variáveis preditoras e a variável-alvo.

As 16 features selecionadas na etapa anterior permanecerão em `X`, enquanto `in_alfabetizado` será utilizado como target (`y`).

Essa separação será realizada de forma idêntica nos conjuntos de treino, validação e teste, preservando as mesmas partições definidas anteriormente. As variáveis auxiliares permanecerão em estruturas separadas e não serão incluídas em `X`.

In [0]:
# Objetivo:
#
# Separar as features e o target nos conjuntos
# de treino, validação e teste.
#
# Justificativa:
#
# A modelagem supervisionada exige que as
# variáveis preditoras sejam separadas da
# variável-alvo utilizada para treinamento
# e avaliação dos modelos.
#
# A separação será feita de forma consistente
# nos três conjuntos já definidos.
#
# Ação:
#
# Remove o target dos DataFrames para formar
# X e extrai in_alfabetizado para formar y.

X_treino = df_treino.drop(
    columns="in_alfabetizado"
)

y_treino = df_treino[
    "in_alfabetizado"
].copy()


X_validacao = df_validacao.drop(
    columns="in_alfabetizado"
)

y_validacao = df_validacao[
    "in_alfabetizado"
].copy()


X_teste = df_teste.drop(
    columns="in_alfabetizado"
)

y_teste = df_teste[
    "in_alfabetizado"
].copy()


pd.Series({
    "X_treino": X_treino.shape,
    "y_treino": y_treino.shape,
    "X_validacao": X_validacao.shape,
    "y_validacao": y_validacao.shape,
    "X_teste": X_teste.shape,
    "y_teste": y_teste.shape
})

## 3. Integração do pré-processamento reutilizável

A arquitetura de pré-processamento definida e validada na etapa anterior foi externalizada para o módulo `src/preprocessing.py`.

Essa separação permite reutilizar a mesma lógica de engenharia de atributos e transformação dos dados durante a modelagem, evitando duplicação de código entre notebooks e mantendo o pré-processamento integrado ao fluxo de Machine Learning.

O módulo disponibiliza a função `criar_pre_processamento()`, capaz de construir a arquitetura com ou sem padronização das variáveis numéricas, de acordo com as necessidades da família de modelos utilizada.

### 3.1 Disponibilização do módulo Python no Databricks

Para que a importação de `src.preprocessing` seja independente da localização do notebook no Workspace, o arquivo reutilizável deve ser disponibilizado na estrutura oficial do projeto.

Neste projeto, os arquivos devem ser organizados da seguinte forma:

```text
/Volumes/workspace/default/vol_trio_drive/
└── projetos/fiap/tech_challenge_fase3/
    └── src/
        ├── __init__.py
        └── preprocessing.py
```

O arquivo `__init__.py` pode permanecer vazio. Sua presença identifica `src` como pacote Python. O arquivo `preprocessing.py` deve conter a versão que disponibiliza o parâmetro opcional `saida_densa`, necessário para o `HistGradientBoostingClassifier`.

A célula seguinte adiciona a raiz do projeto ao caminho de importação do Python, confirma a existência dos dois arquivos e invalida o cache de importações antes da utilização do módulo. Esse procedimento deve ser executado antes de `from src.preprocessing import criar_pre_processamento`.

In [0]:
# Objetivo:
#
# Disponibilizar o pacote src para importação
# no ambiente de execução do Databricks.
#
# Justificativa:
#
# A localização do notebook no Workspace não
# garante que a raiz do projeto esteja presente
# no sys.path do Python. A inclusão explícita
# evita erros como "No module named 'src'".
#
# Ação:
#
# Define a raiz oficial do projeto, valida os
# arquivos do pacote e adiciona essa raiz ao
# caminho de importação.

import importlib
import os
import sys


PROJECT_ROOT = FASE3_ROOT
SRC_PATH = f"{PROJECT_ROOT}/src"
INIT_FILE = f"{SRC_PATH}/__init__.py"
PREPROCESSING_FILE = (
    f"{SRC_PATH}/preprocessing.py"
)


arquivos_modulo = {
    "__init__.py": os.path.isfile(INIT_FILE),
    "preprocessing.py": os.path.isfile(
        PREPROCESSING_FILE
    )
}


if not all(arquivos_modulo.values()):
    arquivos_ausentes = [
        nome
        for nome, existe in arquivos_modulo.items()
        if not existe
    ]

    raise FileNotFoundError(
        "Arquivos ausentes em "
        f"{SRC_PATH}: {arquivos_ausentes}. "
        "Faça o upload de preprocessing.py e "
        "crie um __init__.py vazio antes de "
        "prosseguir."
    )


if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


importlib.invalidate_caches()


print("Módulo de pré-processamento disponível.")
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Arquivo validado: {PREPROCESSING_FILE}")

In [0]:
# Objetivo:
#
# Importar a arquitetura reutilizável de
# pré-processamento criada na etapa anterior.
#
# Justificativa:
#
# A centralização dessa lógica em um módulo
# Python evita duplicação de código e garante
# que os modelos utilizem a mesma estratégia
# de preparação dos dados.
#
# Ação:
#
# Importa a função responsável pela construção
# da arquitetura de pré-processamento e cria
# uma instância para verificar sua integração
# com o Notebook 08.

from src.preprocessing import criar_pre_processamento


pre_processamento = criar_pre_processamento(
    padronizar=False
)

pre_processamento

## 4. Estratégia de avaliação dos modelos

A modelagem será tratada como um problema de classificação binária supervisionada, tendo como objetivo identificar alunos com risco de não alfabetização.

Embora a codificação original do target `in_alfabetizado` seja:

- `0` — não alfabetizado;
- `1` — alfabetizado;

a classe `0` será considerada a classe positiva de interesse para fins de avaliação, pois representa justamente a condição que o projeto busca identificar.

### 4.1 Custo dos erros

Sob essa perspectiva, um falso negativo ocorre quando um aluno realmente não alfabetizado é classificado pelo modelo como alfabetizado.

Esse erro possui especial relevância para o problema, pois representa um aluno em situação de risco que não seria identificado para eventual priorização.

Por outro lado, um falso positivo ocorre quando um aluno alfabetizado é classificado como não alfabetizado. Nesse caso, o modelo produz um alerta desnecessário, podendo direcionar atenção ou recursos para um aluno que não pertence ao grupo de risco.

Dessa forma, a estratégia de avaliação buscará reduzir falsos negativos sem ignorar o custo associado à geração excessiva de falsos positivos.

### 4.2 Métricas de avaliação

O **recall da classe não alfabetizado (`0`)** terá papel central na avaliação dos modelos, pois mede a proporção dos alunos realmente não alfabetizados que foram corretamente identificados.

Entretanto, o recall não será analisado isoladamente. Um modelo que classifique indiscriminadamente grande parte dos alunos como pertencentes ao grupo de risco pode apresentar recall elevado e, ao mesmo tempo, gerar quantidade excessiva de falsos positivos.

Por esse motivo, a avaliação considerará também a precisão da classe de interesse e o F1-score, permitindo analisar o equilíbrio entre identificação dos alunos em risco e qualidade dos alertas produzidos.

A matriz de confusão será utilizada para traduzir as métricas em quantidades concretas de alunos corretamente e incorretamente classificados.

Métricas complementares de discriminação, como PR-AUC e ROC-AUC, também serão consideradas durante a comparação dos modelos. Outras medidas de desempenho e calibração poderão ser incorporadas nas etapas posteriores de validação conforme a necessidade da análise.

Sempre que uma métrica binária depender da definição explícita da classe positiva, a classe `0` será informada como referência, evitando que o comportamento padrão das funções seja interpretado incorretamente como avaliação da classe de interesse do projeto.

### 4.3 Regra para seleção

A seleção do modelo não será baseada em uma única métrica isolada.

Será priorizado um modelo capaz de identificar adequadamente os alunos não alfabetizados, com especial atenção à redução dos falsos negativos, mantendo simultaneamente um nível de precisão que evite volume excessivo de alertas desnecessários.

As decisões de seleção, ajuste de hiperparâmetros e definição do limiar de classificação serão realizadas utilizando os dados de treino e validação. O conjunto de teste permanecerá isolado até que essas decisões estejam congeladas.

## 5. Baseline mínimo

Antes do treinamento dos modelos preditivos, será estabelecido um baseline mínimo de desempenho.

O baseline servirá como referência para verificar se os modelos supervisionados são capazes de superar uma estratégia ingênua de classificação.

Antes de sua construção, será analisada a distribuição do target no conjunto de treino, permitindo compreender o balanceamento entre alunos alfabetizados e não alfabetizados.

Essa análise será realizada exclusivamente sobre `y_treino`, sem utilizar os conjuntos de validação e teste para decisões de desenvolvimento.

In [0]:
# Objetivo:
#
# Analisar a distribuição da variável-alvo
# no conjunto de treino.
#
# Justificativa:
#
# Antes da construção do baseline e da eventual
# aplicação de técnicas para desbalanceamento,
# é necessário conhecer a frequência e a proporção
# das classes presentes no target.
#
# A classe 0 representa os alunos não alfabetizados
# e constitui a classe positiva de interesse
# para o projeto.
#
# Ação:
#
# Calcula a quantidade e a proporção percentual
# das classes presentes em y_treino.

distribuicao_target = pd.DataFrame({
    "quantidade": y_treino.value_counts().sort_index(),
    "percentual": (
        y_treino
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

distribuicao_target.index = [
    "0 - Não alfabetizado",
    "1 - Alfabetizado"
]

distribuicao_target

### 5.1 Construção e treinamento do baseline ingênuo

O primeiro modelo utilizado será o `DummyClassifier`, que servirá como baseline mínimo para a comparação dos modelos posteriores.

Será adotada a estratégia `most_frequent`, na qual o classificador aprende apenas qual é a classe mais frequente no conjunto de treino e passa a utilizá-la como previsão.

Esse modelo não busca aprender relações entre as features e o target. Sua função é estabelecer uma referência mínima de desempenho que os modelos supervisionados deverão superar.

Mesmo tratando-se de um classificador ingênuo, o modelo será integrado à arquitetura de pré-processamento por meio de uma `Pipeline` do Scikit-learn, mantendo o mesmo padrão que será utilizado nos modelos posteriores.

Como o `DummyClassifier` não depende da escala das variáveis, a padronização numérica permanecerá desativada nesta pipeline.

In [0]:
# Objetivo:
#
# Construir e treinar o baseline mínimo
# utilizando o DummyClassifier.
#
# Justificativa:
#
# O baseline ingênuo estabelece uma referência
# mínima de desempenho para os modelos que serão
# avaliados posteriormente.
#
# A estratégia most_frequent prevê sempre a classe
# mais frequente aprendida no conjunto de treino.
#
# Mesmo para o baseline, o pré-processamento será
# mantido integrado ao estimador por meio de uma
# Pipeline do Scikit-learn.
#
# Ação:
#
# Constrói uma nova arquitetura de pré-processamento,
# integra o DummyClassifier e ajusta a pipeline
# completa exclusivamente sobre o conjunto de treino.

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline


pipeline_dummy = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False
            )
        ),
        (
            "modelo",
            DummyClassifier(
                strategy="most_frequent"
            )
        )
    ]
)


pipeline_dummy.fit(
    X_treino,
    y_treino
)

### 5.2 Avaliação do baseline no conjunto de validação

Após o treinamento exclusivamente sobre o conjunto de treino, o baseline será avaliado no conjunto de validação.

O conjunto de validação contém municípios não utilizados no ajuste da pipeline, permitindo observar o comportamento do baseline fora dos grupos utilizados no treinamento.

Nesta primeira avaliação serão calculadas métricas de classificação sob a perspectiva da classe de interesse `0` — alunos não alfabetizados.

Serão analisados:

- recall da classe não alfabetizado;
- precisão da classe não alfabetizado;
- F1-score da classe não alfabetizado;
- acurácia;
- balanced accuracy;
- matriz de confusão.

O conjunto de teste permanecerá isolado e não participará desta etapa.

In [0]:
# Objetivo:
#
# Avaliar o baseline ingênuo no conjunto
# de validação.
#
# Justificativa:
#
# O conjunto de validação permite medir o
# desempenho do modelo em municípios que não
# participaram do treinamento.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de
# classificação do baseline.

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)


y_pred_dummy = pipeline_dummy.predict(
    X_validacao
)


metricas_dummy = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_dummy
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_dummy
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_dummy,
        pos_label=0,
        zero_division=0
    )
})


metricas_dummy

### 5.3 Matriz de confusão do baseline

A matriz de confusão será utilizada para analisar os erros e acertos do baseline em termos absolutos, permitindo traduzir as métricas de classificação em quantidade de alunos.

Como a classe positiva de interesse do projeto é `0` — não alfabetizado — a interpretação da matriz seguirá essa perspectiva.

Serão identificados:

- verdadeiro positivo (VP): aluno não alfabetizado corretamente identificado como não alfabetizado;
- falso negativo (FN): aluno não alfabetizado incorretamente classificado como alfabetizado;
- falso positivo (FP): aluno alfabetizado incorretamente classificado como não alfabetizado;
- verdadeiro negativo (VN): aluno alfabetizado corretamente identificado como alfabetizado.

Entre esses erros, o falso negativo possui especial relevância para o projeto, pois representa um aluno realmente não alfabetizado que deixou de ser identificado pelo modelo.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# do baseline no conjunto de validação.
#
# Justificativa:
#
# A matriz de confusão permite traduzir as
# métricas em quantidades concretas de alunos
# classificados correta e incorretamente.
#
# Como a classe positiva de interesse é 0,
# a interpretação será realizada sob a
# perspectiva dos alunos não alfabetizados.
#
# Ação:
#
# Calcula a matriz de confusão com ordem
# explícita das classes e extrai VP, FN,
# FP e VN.

from sklearn.metrics import confusion_matrix


matriz_dummy = confusion_matrix(
    y_validacao,
    y_pred_dummy,
    labels=[0, 1]
)


vp = matriz_dummy[0, 0]
fn = matriz_dummy[0, 1]
fp = matriz_dummy[1, 0]
vn = matriz_dummy[1, 1]


pd.Series({
    "VP_nao_alfabetizado": vp,
    "FN_nao_alfabetizado": fn,
    "FP_nao_alfabetizado": fp,
    "VN_nao_alfabetizado": vn
})

## 6. Baseline explicável — Regressão Logística

Após o estabelecimento do baseline ingênuo, será construída uma Regressão Logística como primeiro modelo capaz de aprender relações entre as features e o target.

A Regressão Logística constitui um baseline explicável e fornece uma referência linear para comparação com modelos mais complexos nas etapas posteriores.

Como o algoritmo é sensível à escala das variáveis e utiliza regularização, o pré-processamento será configurado com padronização das features numéricas.

Também será utilizada inicialmente a estratégia `class_weight="balanced"`, atribuindo pesos inversamente proporcionais à frequência das classes durante o treinamento.

Essa configuração busca aumentar a importância relativa da classe menos frequente — alunos não alfabetizados — sem modificar fisicamente a distribuição dos dados de treino.

O desempenho obtido será posteriormente comparado com outras configurações e modelos, considerando especialmente o recall da classe não alfabetizado e seu trade-off com precisão e falsos positivos.

In [0]:
# Objetivo:
#
# Construir e treinar o primeiro baseline
# explicável utilizando Regressão Logística.
#
# Justificativa:
#
# A Regressão Logística permite estabelecer
# uma referência linear e interpretável para
# comparação com modelos mais complexos.
#
# Como o algoritmo utiliza regularização e é
# sensível à escala das variáveis, a padronização
# será ativada no pré-processamento.
#
# O balanceamento das classes será considerado
# inicialmente para aumentar a importância
# relativa da classe menos frequente durante
# o treinamento.
#
# Ação:
#
# Constrói uma pipeline completa contendo
# pré-processamento com padronização e
# Regressão Logística balanceada e realiza
# o ajuste exclusivamente no conjunto de treino.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


pipeline_logistica_balanceada = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=True
            )
        ),
        (
            "modelo",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


pipeline_logistica_balanceada.fit(
    X_treino,
    y_treino
)

### 6.1 Avaliação da Regressão Logística balanceada

Após o treinamento, a Regressão Logística balanceada será avaliada no conjunto de validação utilizando as mesmas métricas adotadas para o baseline ingênuo.

A manutenção do mesmo protocolo de avaliação permite comparar os modelos de forma consistente.

Como a classe positiva de interesse permanece sendo `0` — não alfabetizado — recall, precisão e F1-score serão calculados explicitamente sob essa perspectiva.

Nesta etapa serão avaliados:

- acurácia;
- balanced accuracy;
- recall da classe não alfabetizado;
- precisão da classe não alfabetizado;
- F1-score da classe não alfabetizado.

O conjunto de teste permanecerá isolado.

In [0]:
# Objetivo:
#
# Avaliar a Regressão Logística balanceada
# no conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas aplicadas
# ao baseline ingênuo permite comparar os
# modelos sob um protocolo consistente.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de
# classificação.

y_pred_logistica_balanceada = (
    pipeline_logistica_balanceada.predict(
        X_validacao
    )
)


metricas_logistica_balanceada = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_logistica_balanceada
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_logistica_balanceada
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_logistica_balanceada,
        pos_label=0,
        zero_division=0
    )
})


metricas_logistica_balanceada

### 6.2 Matriz de confusão da Regressão Logística balanceada

A matriz de confusão será utilizada para traduzir o desempenho da Regressão Logística balanceada em quantidades concretas de alunos.

A análise permitirá verificar quantos alunos não alfabetizados foram corretamente identificados e quantos permaneceram como falsos negativos, além de quantificar os falsos positivos gerados pelo modelo.

Essa interpretação é especialmente importante para avaliar o trade-off observado entre recall e precisão da classe de interesse.

A matriz seguirá a mesma convenção utilizada no baseline, considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Regressão Logística balanceada.
#
# Justificativa:
#
# A matriz de confusão permite traduzir o
# desempenho do modelo em quantidades concretas
# de alunos e compreender o trade-off entre
# falsos negativos e falsos positivos.
#
# A mesma convenção utilizada no baseline será
# mantida, considerando a classe 0 como positiva
# de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_logistica_balanceada = confusion_matrix(
    y_validacao,
    y_pred_logistica_balanceada,
    labels=[0, 1]
)


vp_logistica = matriz_logistica_balanceada[0, 0]
fn_logistica = matriz_logistica_balanceada[0, 1]
fp_logistica = matriz_logistica_balanceada[1, 0]
vn_logistica = matriz_logistica_balanceada[1, 1]


pd.Series({
    "VP_nao_alfabetizado": vp_logistica,
    "FN_nao_alfabetizado": fn_logistica,
    "FP_nao_alfabetizado": fp_logistica,
    "VN_nao_alfabetizado": vn_logistica
})

### 6.3 Comparação com Regressão Logística sem balanceamento

A Regressão Logística balanceada apresentou aumento expressivo na capacidade de identificar alunos não alfabetizados em relação ao baseline ingênuo, porém acompanhado por quantidade relevante de falsos positivos.

Para avaliar especificamente o efeito da ponderação das classes, será treinada uma segunda Regressão Logística sem `class_weight="balanced"`.

As demais configurações serão mantidas iguais, incluindo as features, o conjunto de treino, o pré-processamento com padronização e os parâmetros do estimador.

Dessa forma, a comparação entre as duas versões permitirá observar empiricamente o efeito da ponderação das classes sobre recall, precisão, F1-score e demais métricas de validação.

In [0]:
# Objetivo:
#
# Construir e treinar uma Regressão Logística
# sem balanceamento automático das classes.
#
# Justificativa:
#
# A comparação com a versão balanceada permitirá
# avaliar o efeito específico de class_weight
# sobre o comportamento do modelo.
#
# Para tornar a comparação consistente, todas
# as demais configurações serão mantidas iguais.
#
# Ação:
#
# Constrói uma nova pipeline com o mesmo
# pré-processamento utilizado anteriormente,
# mas sem ponderação das classes, e realiza
# o ajuste exclusivamente no conjunto de treino.

pipeline_logistica_sem_balanceamento = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=True
            )
        ),
        (
            "modelo",
            LogisticRegression(
                class_weight=None,
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


pipeline_logistica_sem_balanceamento.fit(
    X_treino,
    y_treino
)

### 6.4 Avaliação da Regressão Logística sem balanceamento

Após o treinamento, a Regressão Logística sem balanceamento será avaliada no mesmo conjunto de validação e com as mesmas métricas utilizadas na versão balanceada.

Como todas as demais configurações foram mantidas, essa comparação permitirá observar o efeito da ponderação das classes sobre o comportamento do modelo.

A análise continuará priorizando a classe `0` — não alfabetizado — com especial atenção ao recall, à precisão e ao F1-score.

O conjunto de teste permanecerá isolado.

In [0]:
# Objetivo:
#
# Avaliar a Regressão Logística sem
# balanceamento no conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas e do mesmo
# conjunto de validação permite comparar esta
# configuração diretamente com a versão
# balanceada.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as mesmas métricas utilizadas na
# avaliação da Regressão Logística balanceada.

y_pred_logistica_sem_balanceamento = (
    pipeline_logistica_sem_balanceamento.predict(
        X_validacao
    )
)


metricas_logistica_sem_balanceamento = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_logistica_sem_balanceamento,
        pos_label=0,
        zero_division=0
    )
})


metricas_logistica_sem_balanceamento

### 6.5 Matriz de confusão da Regressão Logística sem balanceamento

A matriz de confusão será utilizada para complementar a comparação entre as duas configurações da Regressão Logística.

A análise permitirá quantificar o efeito da retirada do balanceamento das classes sobre verdadeiros positivos, falsos negativos, falsos positivos e verdadeiros negativos.

Como todas as demais condições do experimento foram mantidas, a comparação com a versão balanceada permitirá compreender de forma concreta o trade-off provocado pela ponderação das classes.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Regressão Logística sem balanceamento.
#
# Justificativa:
#
# A comparação com a matriz da versão balanceada
# permitirá observar em quantidades de alunos
# como a ponderação das classes afetou os erros
# e acertos do modelo.
#
# A mesma convenção será mantida, considerando
# a classe 0 como positiva de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_logistica_sem_balanceamento = confusion_matrix(
    y_validacao,
    y_pred_logistica_sem_balanceamento,
    labels=[0, 1]
)


vp_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[0, 0]
)

fn_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[0, 1]
)

fp_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[1, 0]
)

vn_sem_balanceamento = (
    matriz_logistica_sem_balanceamento[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": vp_sem_balanceamento,
    "FN_nao_alfabetizado": fn_sem_balanceamento,
    "FP_nao_alfabetizado": fp_sem_balanceamento,
    "VN_nao_alfabetizado": vn_sem_balanceamento
})

### 6.6 Comparação das configurações da Regressão Logística

As duas configurações da Regressão Logística apresentaram comportamentos substancialmente diferentes no conjunto de validação.

A versão sem balanceamento apresentou maior acurácia e precisão da classe não alfabetizado, porém identificou apenas uma pequena parcela dos alunos pertencentes à classe de interesse.

A utilização de `class_weight="balanced"` aumentou expressivamente o recall da classe não alfabetizado e reduziu a quantidade de falsos negativos, acompanhada, entretanto, por aumento relevante dos falsos positivos e redução da precisão.

A comparação entre as duas configurações evidencia o trade-off entre identificar uma parcela maior dos alunos não alfabetizados e gerar maior quantidade de alertas incorretos.

Esses resultados não determinam, isoladamente, a escolha do modelo final. A decisão deverá considerar os demais modelos candidatos, o custo dos erros, a estabilidade dos resultados e, posteriormente, a definição do limiar de decisão.

In [0]:
# Objetivo:
#
# Consolidar os resultados dos modelos avaliados
# até esta etapa em uma única tabela.
#
# Justificativa:
#
# A comparação lado a lado facilita a análise
# das diferenças entre o baseline ingênuo e as
# duas configurações da Regressão Logística.
#
# Ação:
#
# Organiza as principais métricas obtidas no
# conjunto de validação em uma tabela comparativa.

comparacao_modelos = pd.DataFrame({
    "DummyClassifier": metricas_dummy,
    "Logistica_balanceada": metricas_logistica_balanceada,
    "Logistica_sem_balanceamento": metricas_logistica_sem_balanceamento
})

comparacao_modelos

## 7. Random Forest

Após a avaliação dos baselines, será construído um `RandomForestClassifier` como primeiro modelo não linear do processo de modelagem.

A Random Forest combina múltiplas árvores de decisão construídas com aleatoriedade nos registros e nas features consideradas durante o treinamento, permitindo capturar relações não lineares e interações entre variáveis.

Como modelos baseados em árvores não dependem da escala das variáveis da mesma forma que modelos lineares, o pré-processamento será utilizado sem padronização das features numéricas.

Nesta primeira configuração serão utilizados parâmetros moderados de complexidade e custo computacional, considerando o elevado volume de registros da base.

Também será utilizada inicialmente a ponderação balanceada das classes, mantendo como prioridade a identificação dos alunos não alfabetizados.

Os valores definidos nesta etapa representam uma configuração inicial de referência e não uma otimização de hiperparâmetros. A busca sistemática por configurações mais adequadas será realizada posteriormente.

In [0]:
# Objetivo:
#
# Construir e treinar a primeira Random Forest
# do processo de modelagem.
#
# Justificativa:
#
# A Random Forest permite capturar relações
# não lineares e interações entre as features,
# oferecendo uma comparação com o baseline
# linear estabelecido anteriormente.
#
# Como modelos baseados em árvores não exigem
# padronização das variáveis, o pré-processamento
# será utilizado com padronizar=False.
#
# A ponderação balanceada será considerada
# inicialmente para aumentar a importância
# relativa da classe menos frequente.
#
# Ação:
#
# Constrói uma pipeline completa contendo
# pré-processamento sem padronização e uma
# Random Forest com configuração inicial
# controlada de complexidade.

from sklearn.ensemble import RandomForestClassifier


pipeline_random_forest_balanceada = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False
            )
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                min_samples_leaf=50,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)


pipeline_random_forest_balanceada.fit(
    X_treino,
    y_treino
)

### 7.1 Avaliação da Random Forest balanceada

Após o treinamento, a Random Forest balanceada será avaliada no conjunto de validação utilizando as mesmas métricas aplicadas aos modelos anteriores.

A manutenção do mesmo protocolo de avaliação permite comparar diretamente o comportamento da Random Forest com o baseline ingênuo e com as configurações da Regressão Logística.

Como a classe `0` — não alfabetizado — permanece como classe positiva de interesse, serão priorizados o recall, a precisão e o F1-score dessa classe, além da acurácia e da acurácia balanceada.

O conjunto de teste permanecerá isolado nesta etapa.

In [0]:
# Objetivo:
#
# Avaliar a Random Forest balanceada no
# conjunto de validação.
#
# Justificativa:
#
# A utilização das mesmas métricas aplicadas
# aos modelos anteriores permite uma comparação
# consistente entre os diferentes candidatos.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas de avaliação.

y_pred_random_forest_balanceada = (
    pipeline_random_forest_balanceada.predict(
        X_validacao
    )
)


metricas_random_forest_balanceada = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_random_forest_balanceada
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_random_forest_balanceada
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_random_forest_balanceada,
        pos_label=0,
        zero_division=0
    )
})


metricas_random_forest_balanceada

### 7.2 Matriz de confusão da Random Forest balanceada

A matriz de confusão será utilizada para traduzir o desempenho da Random Forest balanceada em quantidades concretas de alunos.

A análise permitirá verificar quantos alunos não alfabetizados foram corretamente identificados e quantos permaneceram como falsos negativos, além de quantificar os falsos positivos e verdadeiros negativos produzidos pelo modelo.

Os resultados também permitirão comparar diretamente o comportamento da Random Forest com o da Regressão Logística balanceada.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# da Random Forest balanceada.
#
# Justificativa:
#
# A matriz de confusão permite traduzir as
# métricas do modelo em quantidades concretas
# de alunos e comparar seu comportamento com
# os modelos avaliados anteriormente.
#
# A mesma convenção será mantida, considerando
# a classe 0 como positiva de interesse.
#
# Ação:
#
# Calcula a matriz de confusão no conjunto de
# validação e extrai VP, FN, FP e VN.

matriz_random_forest_balanceada = confusion_matrix(
    y_validacao,
    y_pred_random_forest_balanceada,
    labels=[0, 1]
)


vp_random_forest = (
    matriz_random_forest_balanceada[0, 0]
)

fn_random_forest = (
    matriz_random_forest_balanceada[0, 1]
)

fp_random_forest = (
    matriz_random_forest_balanceada[1, 0]
)

vn_random_forest = (
    matriz_random_forest_balanceada[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": vp_random_forest,
    "FN_nao_alfabetizado": fn_random_forest,
    "FP_nao_alfabetizado": fp_random_forest,
    "VN_nao_alfabetizado": vn_random_forest
})

### 7.3 Comparação da Random Forest com os modelos anteriores

Os resultados da primeira configuração da Random Forest serão incorporados à tabela comparativa dos modelos avaliados no conjunto de validação.

A Random Forest balanceada apresentou desempenho muito próximo ao da Regressão Logística balanceada, com pequeno aumento no recall e no F1-score da classe não alfabetizado, acompanhado por pequena redução na precisão.

Na comparação agregada das matrizes de confusão, a Random Forest apresentou 940 verdadeiros positivos adicionais e 940 falsos negativos a menos, ao mesmo tempo em que produziu 1.719 falsos positivos adicionais.

Considerando a proximidade das métricas e o maior custo computacional observado no treinamento da Random Forest, ainda não há evidência suficiente para definir um modelo vencedor.

A comparação será ampliada com os demais modelos candidatos antes da seleção e otimização do modelo final.

In [0]:
# Objetivo:
#
# Incorporar os resultados da Random Forest
# balanceada à tabela comparativa dos modelos.
#
# Justificativa:
#
# A manutenção de uma tabela consolidada permite
# acompanhar a evolução dos candidatos utilizando
# o mesmo conjunto e protocolo de validação.
#
# Ação:
#
# Adiciona as métricas da Random Forest
# balanceada ao DataFrame de comparação
# construído anteriormente.

comparacao_modelos[
    "Random_Forest_balanceada"
] = metricas_random_forest_balanceada


comparacao_modelos

## 8. HistGradientBoostingClassifier

Após a avaliação do baseline ingênuo, da Regressão Logística e da Random Forest, a próxima etapa da modelagem será dedicada ao `HistGradientBoostingClassifier`.

O novo candidato deverá seguir o mesmo protocolo metodológico adotado até aqui:

- utilizar exclusivamente o conjunto de treino durante o ajuste;
- realizar a comparação no conjunto de validação;
- manter o conjunto de teste isolado;
- avaliar as mesmas métricas utilizadas nos modelos anteriores;
- analisar a matriz de confusão considerando a classe `0` — não alfabetizado — como classe positiva de interesse;
- incorporar posteriormente seus resultados à tabela consolidada de comparação dos modelos.

A construção desta etapa será realizada dando continuidade ao processo de modelagem desenvolvido neste notebook.

In [0]:
# Objetivo:
#
# Construir e treinar o modelo
# HistGradientBoostingClassifier.
#
# Justificativa:
#
# O HistGradientBoostingClassifier é um modelo
# de boosting baseado em histogramas, eficiente
# para bases extensas e capaz de representar
# relações não lineares e interações.
#
# Esse estimador exige uma matriz densa. Por
# esse motivo, o pré-processamento será criado
# com saida_densa=True, sem alterar a configuração
# utilizada pelos modelos anteriores.
#
# A ponderação das classes será aplicada por
# meio de sample_weight, preservando a prioridade
# atribuída à classe menos frequente e mantendo
# compatibilidade entre versões do Scikit-learn.
#
# Ação:
#
# Calcula os pesos balanceados, constrói a
# pipeline completa e realiza o ajuste somente
# sobre o conjunto de treino.

from sklearn.ensemble import (
    HistGradientBoostingClassifier
)
from sklearn.utils.class_weight import (
    compute_sample_weight
)


pesos_treino_hist_gradient_boosting = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_treino
    )
)


pipeline_hist_gradient_boosting = Pipeline(
    steps=[
        (
            "pre_processamento",
            criar_pre_processamento(
                padronizar=False,
                saida_densa=True
            )
        ),
        (
            "modelo",
            HistGradientBoostingClassifier(
                loss="log_loss",
                learning_rate=0.08,
                max_iter=150,
                max_leaf_nodes=31,
                min_samples_leaf=100,
                l2_regularization=1.0,
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=10,
                random_state=42
            )
        )
    ]
)


pipeline_hist_gradient_boosting.fit(
    X_treino,
    y_treino,
    modelo__sample_weight=(
        pesos_treino_hist_gradient_boosting
    )
)


pd.Series({
    "iteracoes_executadas": (
        pipeline_hist_gradient_boosting
        .named_steps["modelo"]
        .n_iter_
    ),
    "early_stopping_ativo": (
        pipeline_hist_gradient_boosting
        .named_steps["modelo"]
        .early_stopping
    )
})

### 8.1 Avaliação do HistGradientBoostingClassifier

Após o treinamento, o novo modelo será avaliado no mesmo conjunto de validação e com as mesmas métricas utilizadas nos candidatos anteriores.

A manutenção do protocolo permite comparar o desempenho sem utilizar o conjunto de teste e sem alterar as partições territoriais definidas no Notebook 07.

Como a classe `0` — não alfabetizado — permanece como classe positiva de interesse, recall, precisão e F1-score serão calculados explicitamente sob essa perspectiva.

In [0]:
# Objetivo:
#
# Avaliar o HistGradientBoostingClassifier
# no conjunto de validação.
#
# Justificativa:
#
# O uso do mesmo conjunto e das mesmas métricas
# permite comparar o novo modelo de forma
# consistente com os candidatos anteriores.
#
# Como a classe positiva de interesse é 0,
# recall, precisão e F1 serão calculados
# explicitamente com pos_label=0.
#
# Ação:
#
# Realiza as previsões no conjunto de validação
# e calcula as principais métricas.

y_pred_hist_gradient_boosting = (
    pipeline_hist_gradient_boosting.predict(
        X_validacao
    )
)


metricas_hist_gradient_boosting = pd.Series({
    "accuracy": accuracy_score(
        y_validacao,
        y_pred_hist_gradient_boosting
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validacao,
        y_pred_hist_gradient_boosting
    ),
    "recall_nao_alfabetizado": recall_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0
    ),
    "precision_nao_alfabetizado": precision_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0,
        zero_division=0
    ),
    "f1_nao_alfabetizado": f1_score(
        y_validacao,
        y_pred_hist_gradient_boosting,
        pos_label=0,
        zero_division=0
    )
})


metricas_hist_gradient_boosting

### 8.2 Matriz de confusão do HistGradientBoostingClassifier

A matriz de confusão traduzirá o desempenho do modelo em quantidades concretas de alunos e permitirá comparar verdadeiros positivos, falsos negativos, falsos positivos e verdadeiros negativos com os demais candidatos.

A interpretação continuará considerando `0` — não alfabetizado — como classe positiva de interesse. O falso negativo permanece como erro prioritário, pois representa um aluno não alfabetizado que deixou de ser identificado.

In [0]:
# Objetivo:
#
# Construir e interpretar a matriz de confusão
# do HistGradientBoostingClassifier.
#
# Justificativa:
#
# A matriz permite observar em números absolutos
# os acertos e erros do novo modelo e comparar
# seu comportamento com os demais candidatos.
#
# Ação:
#
# Calcula a matriz com ordem explícita das
# classes e extrai VP, FN, FP e VN.

matriz_hist_gradient_boosting = confusion_matrix(
    y_validacao,
    y_pred_hist_gradient_boosting,
    labels=[0, 1]
)


vp_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[0, 0]
)

fn_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[0, 1]
)

fp_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[1, 0]
)

vn_hist_gradient_boosting = (
    matriz_hist_gradient_boosting[1, 1]
)


pd.Series({
    "VP_nao_alfabetizado": (
        vp_hist_gradient_boosting
    ),
    "FN_nao_alfabetizado": (
        fn_hist_gradient_boosting
    ),
    "FP_nao_alfabetizado": (
        fp_hist_gradient_boosting
    ),
    "VN_nao_alfabetizado": (
        vn_hist_gradient_boosting
    )
})

### 8.3 Comparação com os modelos anteriores

As métricas do `HistGradientBoostingClassifier` serão adicionadas à tabela consolidada construída ao longo do notebook.

Esta comparação ainda representa a configuração inicial dos modelos. Portanto, não deverá ser utilizada isoladamente para declarar um vencedor. A seleção posterior deverá considerar o equilíbrio entre recall e precisão da classe não alfabetizado, o F1-score, a acurácia balanceada, a matriz de confusão, a estabilidade territorial e o custo computacional.

In [0]:
# Objetivo:
#
# Incorporar o HistGradientBoostingClassifier
# à comparação consolidada dos modelos.
#
# Justificativa:
#
# A tabela lado a lado permite verificar o
# comportamento de todos os candidatos sob
# o mesmo protocolo de validação.
#
# Ação:
#
# Adiciona as métricas do novo modelo e ordena
# as colunas conforme a sequência de construção.

comparacao_modelos[
    "Hist_Gradient_Boosting_balanceado"
] = metricas_hist_gradient_boosting


comparacao_modelos

### 8.4 Encerramento da construção inicial dos modelos candidatos

Com a inclusão do `HistGradientBoostingClassifier`, o notebook passa a contemplar os quatro modelos candidatos previstos para esta fase inicial:

- `DummyClassifier`, como baseline mínimo;
- `LogisticRegression`, como baseline explicável, com e sem balanceamento;
- `RandomForestClassifier`, como modelo não linear baseado em combinação de árvores;
- `HistGradientBoostingClassifier`, como modelo de boosting eficiente para bases extensas.

Todos os candidatos foram integrados ao mesmo módulo de pré-processamento, treinados exclusivamente sobre o conjunto de treino e avaliados sobre o conjunto de validação. O conjunto de teste permanece isolado.

A configuração do HistGradientBoosting apresentada nesta seção constitui um ponto inicial controlado, e não uma otimização final. As próximas etapas deverão aprofundar a comparação, a validação estatística, a seleção de hiperparâmetros e a definição do limiar de decisão antes da avaliação final no conjunto de teste.